In [ ]:
# download the weaviate client
%pip install -U weaviate-client

In [ ]:
import weaviate, os
from weaviate.config import AdditionalConfig, Timeout
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve environment variables
CLUSTER_URL = os.getenv("CLUSTER_URL")
API_KEY = os.getenv("API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Connect to Weaviate
client = weaviate.connect_to_weaviate_cloud(
	cluster_url=CLUSTER_URL,
	auth_credentials=weaviate.auth.AuthApiKey(API_KEY),
	headers={
		"X-OpenAI-Api-Key": OPENAI_API_KEY,
		"X-Cohere-Api-Key": COHERE_API_KEY,
        "X-Goog-Api-Key": GOOGLE_API_KEY
	},
	additional_config=AdditionalConfig(
		timeout=Timeout(init=30, query=60, insert=120)
	)
)

ready = client.is_ready()
server_version = client.get_meta()["version"]
client_version = weaviate.__version__
live = client.is_live()
connected = client.is_connected()

print(f"Weaviate Ready: {ready}")
print(f"Weaviate Client Version: {client_version}")
print(f"Weaviate Server Version: {server_version}")
print(f"Weaviate Live: {client.is_live()}")
print(f"Client Connected: {connected}")

In [ ]:
# Function to update an object in Weaviate
def update_object(uuid):
    try:
        update_object = client.collections.use("<COLLECTION_NAME>")  # Replace with your collection name
        update_object.data.update(
            uuid=uuid,
            properties={
                "<PROPERTY_NAME>": "<NEW_VALUE>"  # Replace with the property you want to update and its new value,
                # Whatever other properties you want to update can be added here, will only update the ones you specify
            }
        )
        return True, f"Object with UUID '{uuid}' updated successfully."
    except Exception as e:
        return False, f"Error updating object with UUID '{uuid}': {str(e)}"


In [ ]:
# Update shards status
coll = client.collections.use("<COLLECTION_NAME>")  # Replace with your collection name

shards = coll.config.update_shards(
    status="READONLY",
    shard_names=["<SHARD_NAME>"]  # The names (List[str]) of the shard to update (or a shard name)
)

print(shards)

In [ ]:
# Update Replication Config
from weaviate.classes.config import Reconfigure, ReplicationDeletionStrategy

def update_replication_config(
	client,
	collection_name,
	async_enabled=None,
	deletion_strategy=None
):
	collection = client.collections.use(collection_name)
	collection.config.update(
		replication_config=Reconfigure.replication(
			async_enabled=async_enabled,
			deletion_strategy=deletion_strategy
		)
	)

def update_all_collections_replication(
	client,
	async_enabled=False,
	deletion_strategy=None
):
	"""Fetch all collections and update their replication configuration."""
	try:
		# Get all collection names
		all_collections = client.collections.list_all()
		collection_names = list(all_collections.keys())
		
		print(f"Found {len(collection_names)} collections to update.")
		
		# Track results
		successful = []
		failed = []
		
		# Loop through each collection and update
		for collection_name in collection_names:
			try:
				update_replication_config(
					client,
					collection_name,
					async_enabled=async_enabled,
					deletion_strategy=deletion_strategy
				)
				successful.append(collection_name)
				print(f"✓ Updated: {collection_name}")
			except Exception as e:
				failed.append((collection_name, str(e)))
				print(f"✗ Failed: {collection_name} - {str(e)}")
		
		# Print summary
		print(f"\n{'='*60}")
		print(f"Summary: {len(successful)} successful, {len(failed)} failed")
		print(f"{'='*60}")
		
		if failed:
			print("\nFailed collections:")
			for name, error in failed:
				print(f"  - {name}: {error}")
		
		return successful, failed
		
	except Exception as e:
		print(f"Error fetching collections: {str(e)}")
		return [], []

# Execute the update for all collections
successful, failed = update_all_collections_replication(
	client,
	async_enabled=True,
	deletion_strategy=ReplicationDeletionStrategy.TIME_BASED_RESOLUTION,
)


In [ ]:
# Update the vector index configuration for a collection to increase the threshold
from weaviate.classes.config import Reconfigure

collection = client.collections.use("<COLLECTION_NAME>")
result = collection.config.update(
    vectorizer_config=Reconfigure.VectorIndex.dynamic(
        threshold=100000
    )
)
print(result)

In [ ]:
# Function to update object properties
def update_object_properties(client, collection_name, uuid, properties, tenant=None):
    try:
        collection = client.collections.use(collection_name)
        if tenant:
            collection = collection.with_tenant(tenant)
        
        collection.data.update(
            uuid=uuid,
            properties=properties
        )
        return True
    except Exception as e:
        raise Exception(f"Failed to update object: {str(e)}")


collection_name = "<COLLECTION_NAME>"
object_uuid = "<OBJECT_UUID>"
new_properties = {
    "<PROPERTY_NAME>": [
    { "<SUB_PROPERTY_NAME>": "<SUB_PROPERTY_VALUE>" },
    { "<SUB_PROPERTY_NAME>": "SUB_PROPERTY_VALUE" }
    ]
}

# Calling the method to update the object
result = update_object_properties(client, collection_name, object_uuid, new_properties)
print("Update successful:", result)

In [ ]:
# Update Collection Description
from weaviate.classes.config import Reconfigure

# Get the Article collection object
articles = client.collections.use("<COLLECTION_NAME>")

# Update the collection configuration to remove stopwords "a" and "the"
articles.config.update(
    description="<NEW_DESCRIPTION>",
)

In [ ]:
# Update the collection Inverted Index Configuration
from weaviate.classes.config import Reconfigure, StopwordsPreset

def update_inverted_index_config(
    client,
    collection_name,
    bm25_b=None,
    bm25_k1=None,
    cleanup_interval_seconds=None,
    stopwords_preset=None,
    stopwords_additions=None,
    stopwords_removals=None
):
    collection = client.collections.use(collection_name)
    collection.config.update(
        inverted_index_config=Reconfigure.inverted_index(
            bm25_b=bm25_b,
            bm25_k1=bm25_k1,
            cleanup_interval_seconds=cleanup_interval_seconds,
            stopwords_preset=stopwords_preset,
            stopwords_additions=stopwords_additions,
            stopwords_removals=stopwords_removals
        )
    )

update_inverted_index_config(
    client,
    "<COLLECTION_NAME>",
    bm25_b=0.5,
    bm25_k1=1.8,
    cleanup_interval_seconds=90,
    stopwords_preset=StopwordsPreset.EN,
    stopwords_additions=["<NEW_STOPWORD_1>", "<NEW_STOPWORD_2>", "<NEW_STOPWORD_3>"],
    stopwords_removals=["<STOPWORD_1>", "<STOPWORD_2>"]
)

In [ ]:
# Update Filter Strategy for a collection
from weaviate.classes.config import (
    Reconfigure,
    VectorFilterStrategy,
)

articles = client.collections.use("<COLLECTION_NAME>")  # Replace with your collection name

articles.config.update(
    vector_config=Reconfigure.Vectors.update(
        name="default",
        vector_index_config=Reconfigure.VectorIndex.hnsw(
            filter_strategy=VectorFilterStrategy.ACORN
        ),
    ),
)

In [ ]:
# Bulk update filter strategy across all collections
from weaviate.classes.config import (
    Reconfigure,
    VectorFilterStrategy,
)

def update_filter_strategy_all_collections(
    client,
    target_vector_name="default",
    filter_strategy=VectorFilterStrategy.ACORN,
    collection_names=None,
):
    """Update vector filter strategy for all collections (or a provided subset)."""
    all_collections = client.collections.list_all()
    discovered_names = list(all_collections.keys())
    names_to_update = collection_names or discovered_names

    print(f"Discovered collections: {len(discovered_names)}")
    print(f"Target collections to update: {len(names_to_update)}")

    successful = []
    failed = []

    for collection_name in names_to_update:
        try:
            collection = client.collections.use(collection_name)
            collection.config.update(
                vector_config=Reconfigure.Vectors.update(
                    name=target_vector_name,
                    vector_index_config=Reconfigure.VectorIndex.hnsw(
                        filter_strategy=filter_strategy
                    ),
                ),
            )
            successful.append(collection_name)
            print(f"✓ Updated: {collection_name}")
        except Exception as e:
            failed.append((collection_name, str(e)))
            print(f"✗ Failed: {collection_name} - {str(e)}")

    # Build the final failure list by comparing target collections vs successful updates.
    successful_set = set(successful)
    failed_error_map = {name: err for name, err in failed}
    not_updated = [name for name in names_to_update if name not in successful_set]

    print(f"\nSummary: {len(successful)} successful, {len(not_updated)} not updated")

    print("\nUpdated collections:")
    if successful:
        for name in successful:
            print(f"- {name}")
    else:
        print("- None")

    print("\nNot updated collections (with reason if available):")
    if not_updated:
        for name in not_updated:
            print(f"- {name}: {failed_error_map.get(name, 'No error captured')} ")
    else:
        print("- None")

    return successful, not_updated

# Run for all collections
successful, not_updated = update_filter_strategy_all_collections(
    client=client,
    target_vector_name="default",
    filter_strategy=VectorFilterStrategy.ACORN,
)